# 15 — Sentiment Analysis: Lexicon vs Learned Model

**Learning objective.** Compare a transparent lexicon baseline with a supervised text classifier and inspect disagreement.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


In [2]:
positive={'excellent','amazing','love','premium','great','reliable','sharp','smooth','perfectly','fast'}
negative={'drains','overheats','terrible','crashes','slow','regret','poor','disappointing'}
def lexicon_score(text):
    tok=re.findall(r"\b\w+\b",text.lower())
    s=sum(t in positive for t in tok)-sum(t in negative for t in tok)
    return 'positive' if s>0 else 'negative' if s<0 else 'neutral'
df=pd.read_csv(DATA/'sentiment_reviews.csv')
df['lexicon']=df.text.map(lexicon_score)
print('Lexicon accuracy:',round((df.label==df.lexicon).mean(),3))
df[['text','label','lexicon']].head(8)

Lexicon accuracy: 1.0


                                                     text     label   lexicon
0  The battery life is excellent and the camera is sharp.  positive  positive
1                  Amazing screen and smooth performance.  positive  positive
2             I love the build quality and fast charging.  positive  positive
3            The phone feels premium and works perfectly.  positive  positive
4                Great value for money and very reliable.  positive  positive
5    The battery drains quickly and the device overheats.  negative  negative
6                   Terrible camera quality in low light.  negative  negative
7          The app crashes often and performance is slow.  negative  negative

In [3]:
for s in ['not good at all','not terrible','great camera but awful battery']:
    print(s,'->',lexicon_score(s))

not good at all -> neutral
not terrible -> negative
great camera but awful battery -> positive


Sentiment is compositional: negation, sarcasm, contrast and domain meaning break simple word-count approaches. Always define the label taxonomy (binary, ternary, aspect-level, emotion) before choosing a model.

---
    ## Production takeaways
    - Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
    - Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
    - Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Explain sentiment label design
- Identify negation and composition as baseline failure modes